# 02 — Preprocessing and Feature Extraction

**Project 2.2 — Cross-Modal Retrieval: Mobile App Screens and Text Summaries**

Tess Santhosh (a2910201) · COMP 1030 Multimodal Data Analytics, Adelaide University

This notebook turns the raw screens/captions into cached ResNet-50 (image) and BERT (text) features that notebooks 03–05 train and evaluate on. Also cleans captions and confirms the data split.


From notebook 01: the official train/val/test split is app-disjoint, so no re-split is needed. 0 corrupt images and 0 empty captions were found, so only light whitespace cleaning is needed here.

**Outputs:**
- `data/features/resnet50_{split}.pt`, `bert_{split}.pt`, `eval_manifest_{split}.csv`
- `results/tables/preprocessing_summary.csv`, `backbone_parameter_counts.csv`

## Setup

In [ ]:
#repo setup
import os, sys, subprocess

REPO_URL = "https://github.com/tesssanthosh/cross-modal-retrieval-2.2.git"
REPO_DIR = "/content/cross-modal-retrieval-2.2"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Working directory:", os.getcwd())


Working directory: /content/cross-modal-retrieval-2.2


In [ ]:
!pip install -q -r requirements.txt
!pip install -q datasets


In [ ]:
import random
import time

import numpy as np
import pandas as pd
import torch

from src.utils import (
    count_parameters,
    build_resnet50_backbone, build_bert_backbone,
    extract_resnet50_features, extract_bert_features,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected change runtime type to T4 GPU, then re-run this cell. ")

os.makedirs("data/features", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)


Device: cuda
GPU: Tesla T4


## Load the dataset and confirm the split

Same load as notebook 01 it is repeated here so this notebook runs standalone.

In [ ]:
from datasets import load_dataset

ds = load_dataset("rootsautomation/RICO-Screen2Words")
for name, d in ds.items():
    print(f"{name}: {len(d)} screens")


README.md:   0%|          | 0.00/6.13k [00:00<?, ?B/s]

data/train-00000-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  216MB            

data/train-00000-of-00008.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  219MB            

data/train-00001-of-00008.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  225MB            

... (19 outputs left out)

Generating train split:   0%|          | 0/15743 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/2364 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4310 [00:00<?, ? examples/s]

train: 15743 screens
val: 2364 screens
test: 4310 screens


## Clean captions

Cleaning here is minimal, just collapse whitespace and strip leading/trailing spaces because captions were already confirmed clean in notebook 01. We don’t lowercase, strip punctuation, or remove stopwords, BERT’s tokenizer handles casing and punctuation itself, and that kind of cleaning would throw away information a pretrained language model can use.


In [ ]:
import re

def clean_caption(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text

#creating one row per (screenId, caption) pair
def build_flat_captions(split_dataset) -> pd.DataFrame:
    rows = []
    for row in split_dataset:
        screen_id = row["screenId"]
        app = row["app_package_name"]
        cat = row["category"]
        for idx, cap in enumerate(row["captions"]):
            cap_clean = clean_caption(cap)
            if cap_clean:  # dropping anything that cleans down to empty
                rows.append({"screenId": screen_id, "caption_idx": idx, "caption": cap_clean,
                             "app_package_name": app, "category": cat})
    return pd.DataFrame(rows)

flat = {split_name: build_flat_captions(ds[split_name]) for split_name in ds.keys()}
for split_name, df in flat.items():
    print(f"{split_name}: {len(df)} (screenId, caption) rows from {df['screenId'].nunique()} screens")


train: 78715 (screenId, caption) rows from 15743 screens
val: 11820 (screenId, caption) rows from 2364 screens
test: 21550 (screenId, caption) rows from 4310 screens


## Image feature extraction (frozen ResNet-50)

One 2048-d feature vector per screen  and not per caption, globally average-pooled from frozen ResNet-50. The model and its preprocessing both come from src/utils.py so notebooks 03/04/05 extract features identically.


In [ ]:
resnet, resnet_preprocess = build_resnet50_backbone(DEVICE)
print(f"ResNet-50 backbone: {count_parameters(resnet):,} parameters (frozen)")


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 192MB/s]


ResNet-50 backbone: 23,508,032 parameters (frozen)


In [ ]:
image_feature_cache = {}
image_timing = {}

for split_name in ds.keys():
    split_data = ds[split_name]
    screen_ids = split_data["screenId"]
    images = split_data["image"]  # PIL images are not decoded until actually used

    feats, elapsed = extract_resnet50_features(images, resnet, resnet_preprocess, DEVICE,
                                                batch_size=64)
    image_feature_cache[split_name] = {"screenId": torch.tensor(screen_ids), "features": feats}
    image_timing[split_name] = elapsed

    out_path = f"data/features/resnet50_{split_name}.pt"
    torch.save(image_feature_cache[split_name], out_path)
    print(f"{split_name}: {feats.shape} in {elapsed:.1f}s "
          f"({1000 * elapsed / len(images):.1f} ms/image) -> {out_path}")


ResNet-50 features:   0%|          | 0/246 [00:00<?, ?it/s]

train: torch.Size([15743, 2048]) in 343.8s (21.8 ms/image) -> data/features/resnet50_train.pt


ResNet-50 features:   0%|          | 0/37 [00:00<?, ?it/s]

val: torch.Size([2364, 2048]) in 50.2s (21.2 ms/image) -> data/features/resnet50_val.pt


ResNet-50 features:   0%|          | 0/68 [00:00<?, ?it/s]

test: torch.Size([4310, 2048]) in 93.5s (21.7 ms/image) -> data/features/resnet50_test.pt


In [ ]:
# sanity check: no NaNs, correct shape and one row per screen
for split_name, cache in image_feature_cache.items():
    f = cache["features"]
    assert f.shape[1] == 2048, f"{split_name}: expected 2048-d features, got {f.shape[1]}"
    assert f.shape[0] == len(ds[split_name]), f"{split_name}: row count mismatch"
    assert torch.isfinite(f).all(), f"{split_name}: found NaN/Inf in image features"
print("Image feature sanity checks passed for all splits.")


Image feature sanity checks passed for all splits.


## Text feature extraction (frozen BERT)

One 768-d feature vector per caption, from mean-pooling bert-base-uncased’s last hidden layer. Runs on the flat screenId, caption table, so each of the ~112k captions gets its own row, aligned to flat[split].


In [ ]:
bert, bert_tokenizer = build_bert_backbone(DEVICE)
print(f"BERT backbone: {count_parameters(bert):,} parameters (frozen)")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT backbone: 109,482,240 parameters (frozen)


In [ ]:
text_feature_cache = {}
text_timing = {}

for split_name, df in flat.items():
    texts = df["caption"].tolist()
    feats, elapsed = extract_bert_features(texts, bert, bert_tokenizer, DEVICE, batch_size=128)

    text_feature_cache[split_name] = {
        "screenId": torch.tensor(df["screenId"].values),
        "caption_idx": torch.tensor(df["caption_idx"].values),
        "features": feats,
    }
    text_timing[split_name] = elapsed

    out_path = f"data/features/bert_{split_name}.pt"
    torch.save(text_feature_cache[split_name], out_path)
    print(f"{split_name}: {feats.shape} in {elapsed:.1f}s "
          f"({1000 * elapsed / len(texts):.1f} ms/caption) -> {out_path}")


BERT features:   0%|          | 0/615 [00:00<?, ?it/s]

train: torch.Size([78715, 768]) in 62.8s (0.8 ms/caption) -> data/features/bert_train.pt


BERT features:   0%|          | 0/93 [00:00<?, ?it/s]

val: torch.Size([11820, 768]) in 9.3s (0.8 ms/caption) -> data/features/bert_val.pt


BERT features:   0%|          | 0/169 [00:00<?, ?it/s]

test: torch.Size([21550, 768]) in 17.3s (0.8 ms/caption) -> data/features/bert_test.pt


In [ ]:
for split_name, cache in text_feature_cache.items():
    f = cache["features"]
    assert f.shape[1] == 768, f"{split_name}: expected 768-d features, got {f.shape[1]}"
    assert f.shape[0] == len(flat[split_name]), f"{split_name}: row count mismatch"
    assert torch.isfinite(f).all(), f"{split_name}: found NaN/Inf in text features"
print("Text feature sanity checks passed for all splits.")


Text feature sanity checks passed for all splits.


## Evaluation manifest (one canonical caption per screen)

retrieval_report needs one caption per image a square matrix, so evaluation uses caption index 0 as each screen’s canonical caption and training in 03/04 still uses all 5 per screen for more positive pairs. caption_idx is assigned by the dataset in a fixed order, not randomly, so “caption 0” is reproducible, not an arbitrary pick.


In [ ]:
for split_name, df in flat.items():
    eval_df = df[df["caption_idx"] == 0].reset_index(drop=True)
    eval_df.to_csv(f"data/features/eval_manifest_{split_name}.csv", index=False)
    print(f"{split_name}: {len(eval_df)} eval rows (one per screen) "
          f"-> data/features/eval_manifest_{split_name}.csv")


train: 15743 eval rows (one per screen) -> data/features/eval_manifest_train.csv
val: 2364 eval rows (one per screen) -> data/features/eval_manifest_val.csv
test: 4310 eval rows (one per screen) -> data/features/eval_manifest_test.csv


## Save summary table

In [ ]:
summary_rows = []
for split_name in ds.keys():
    summary_rows.append({
        "split": split_name,
        "n_screens": len(ds[split_name]),
        "n_captions": len(flat[split_name]),
        "image_feature_dim": image_feature_cache[split_name]["features"].shape[1],
        "text_feature_dim": text_feature_cache[split_name]["features"].shape[1],
        "image_extraction_sec": round(image_timing[split_name], 1),
        "text_extraction_sec": round(text_timing[split_name], 1),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("results/tables/preprocessing_summary.csv", index=False)
summary_df


,split,n_screens,n_captions,image_feature_dim,text_feature_dim,image_extraction_sec,text_extraction_sec
0,train,15743,78715,2048,768,343.8,62.8
1,val,2364,11820,2048,768,50.2,9.3
2,test,4310,21550,2048,768,93.5,17.3


In [ ]:
backbone_params_df = pd.DataFrame([
    {"backbone": "ResNet-50 (frozen image encoder)", "parameters": count_parameters(resnet)},
    {"backbone": "BERT-base-uncased (frozen text encoder)", "parameters": count_parameters(bert)},
])
backbone_params_df.to_csv("results/tables/backbone_parameter_counts.csv", index=False)
backbone_params_df


,backbone,parameters
0,ResNet-50 (frozen image encoder),23508032
1,BERT-base-uncased (frozen text encoder),109482240
